# Notebook 05 — Evidence-Grounded Churn Explanation (LLM + Deterministic Verification)

**Project:** Telco Customer Churn Analysis — final stage

This notebook demonstrates an **evidence-grounded LLM explanation workflow**.

```
Customer data
     ↓
XGBoost model          (Notebook 03)
     ↓
SHAP                   (Notebook 04)
     ↓
Structured evidence JSON
     ↓
LLM explanation
     ↓
Independent verification
     ↓
PASS / FLAG
```

**Core principle — the LLM is an explanation layer only, never the source of
truth.**

- The authoritative evidence comes from the trained XGBoost model, actual
  customer features, and the actual SHAP values produced by Notebook 04 and
  stored in `data/evidence/example_customer_evidence.json`.
- The LLM may **summarize and verbalize supplied evidence**. It must not invent
  customer attributes, invent SHAP values, infer unsupported characteristics,
  introduce external facts, or claim causality.
- A **deterministic Python verifier** — independent of the LLM — inspects the
  generated structured output against the authoritative evidence and returns
  `PASS` or `FLAG`.
- An **adversarial test** proves the verifier actually works by feeding it an
  intentionally fabricated explanation that must be flagged.

**Honesty note:** if `OPENAI_API_KEY` is not configured, no real LLM call is
made; the notebook still executes using a clearly labelled deterministic mock,
and every verification test still runs.

## 2. Imports and Environment

- The API key is read **only** from the `OPENAI_API_KEY` environment variable.
  It is never hard-coded, never printed, and never written to any file.
- The model is configurable via `OPENAI_MODEL`, with a documented default.
- `openai` is imported lazily only when a key is present.

In [1]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# --- API configuration (environment variables only) -----------------------
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

print("OPENAI_API_KEY configured:", bool(OPENAI_API_KEY))
print("OPENAI_MODEL:", OPENAI_MODEL)

client = None
if OPENAI_API_KEY:
    from openai import OpenAI

    client = OpenAI(api_key=OPENAI_API_KEY)
    print("OpenAI client created (key loaded from environment).")
else:
    print("No OPENAI_API_KEY found in the environment.")
    print("-> No real LLM call will be made in this run.")
    print("-> Deterministic mock + verification tests will still run and are "
          "clearly labelled as such.")

OPENAI_API_KEY configured: False
OPENAI_MODEL: gpt-4o-mini
No OPENAI_API_KEY found in the environment.
-> No real LLM call will be made in this run.
-> Deterministic mock + verification tests will still run and are clearly labelled as such.


## 3. Load the Authoritative Evidence

The primary evidence source is the JSON produced by Notebook 04:

`data/evidence/example_customer_evidence.json`

The path is resolved robustly from either the repository root or the
`notebooks/` directory.

In [2]:
EVIDENCE_FILE = "data/evidence/example_customer_evidence.json"


def find_evidence_path() -> Path:
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidate = folder / EVIDENCE_FILE
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate {EVIDENCE_FILE}. Run this notebook from the "
        "repository root or from the notebooks/ directory."
    )


EVIDENCE_PATH = find_evidence_path()
print("Resolved evidence path:", EVIDENCE_PATH)

with open(EVIDENCE_PATH) as f:
    raw_evidence = json.load(f)

print("Loaded evidence keys:", list(raw_evidence.keys()))
print("Evidence items:", len(raw_evidence.get("evidence", [])))

Resolved evidence path: /Users/vaibhavvikasranjan/Downloads/telco-churn-analysis/data/evidence/example_customer_evidence.json
Loaded evidence keys: ['customer_index', 'predicted_probability', 'threshold', 'predicted_class', 'base_value', 'evidence']
Evidence items: 45


## 4. Schema Validation

The JSON structure is validated **before** it is trusted. If validation fails,
the problem is reported loudly rather than silently repaired.

In [3]:
def validate_evidence(data):
    '''Return a list of schema/consistency errors (empty list means valid).'''
    errors = []

    required_top = ["customer_index", "predicted_probability",
                    "threshold", "predicted_class", "evidence"]
    for key in required_top:
        if key not in data:
            errors.append(f"missing top-level key: {key}")

    if errors:
        return errors

    prob = data["predicted_probability"]
    thr = data["threshold"]
    if not (isinstance(prob, (int, float)) and 0.0 <= prob <= 1.0):
        errors.append(f"predicted_probability not in [0,1]: {prob!r}")
    if not (isinstance(thr, (int, float)) and 0.0 <= thr <= 1.0):
        errors.append(f"threshold not in [0,1]: {thr!r}")

    evidence = data["evidence"]
    if not isinstance(evidence, list) or len(evidence) == 0:
        errors.append("evidence is missing or empty")

    for i, item in enumerate(evidence):
        for key in ["feature", "value", "shap_value", "direction"]:
            if key not in item:
                errors.append(f"evidence[{i}] missing key: {key}")
                continue
        sv = item.get("shap_value")
        if not (isinstance(sv, (int, float)) and np.isfinite(sv)):
            errors.append(f"evidence[{i}] shap_value is not a finite number: {sv!r}")
        expected = "toward churn" if sv >= 0 else "away from churn"
        if item.get("direction") != expected:
            errors.append(
                f"evidence[{i}] direction {item.get('direction')!r} inconsistent "
                f"with SHAP sign (expected {expected!r})"
            )
    return errors


validation_errors = validate_evidence(raw_evidence)
if validation_errors:
    raise ValueError("Evidence validation failed:\n- " + "\n- ".join(validation_errors))
print("Evidence schema validation passed:")
print("  customer_index:", raw_evidence["customer_index"])
print("  predicted_probability:", raw_evidence["predicted_probability"])
print("  threshold:", raw_evidence["threshold"])
print("  predicted_class:", raw_evidence["predicted_class"])
print("  evidence items:", len(raw_evidence["evidence"]))

Evidence schema validation passed:
  customer_index: 38
  predicted_probability: 0.7300524711608887
  threshold: 0.525
  predicted_class: 1
  evidence items: 45


## 5. Canonical Authoritative Evidence

A single canonical representation is derived from the JSON. Every downstream
check compares against this object; no other source of "truth" is used.

In [4]:
authoritative = {
    "customer_index": raw_evidence["customer_index"],
    "predicted_probability": float(raw_evidence["predicted_probability"]),
    "threshold": float(raw_evidence["threshold"]),
    "predicted_class": int(raw_evidence["predicted_class"]),
    "base_value": float(raw_evidence.get("base_value", 0.0)),
    "features": [
        {
            "feature": item["feature"],
            "value": item["value"],
            "value_display": item.get("value_display", ""),
            "shap_value": float(item["shap_value"]),
            "direction": item["direction"],
        }
        for item in raw_evidence["evidence"]
    ],
}

evidence_df = pd.DataFrame(authoritative["features"])

print("Canonical authoritative evidence:")
print("  customer_index:", authoritative["customer_index"])
print("  predicted_probability:", authoritative["predicted_probability"])
print("  threshold:", authoritative["threshold"])
print("  predicted_class:", authoritative["predicted_class"])
print("  number of features:", len(authoritative["features"]))

assert len(evidence_df) == 45
assert set(evidence_df["feature"]) == {f["feature"] for f in authoritative["features"]}
print("Assertions passed: canonical evidence covers the full Notebook 04 feature set.")

Canonical authoritative evidence:
  customer_index: 38
  predicted_probability: 0.7300524711608887
  threshold: 0.525
  predicted_class: 1
  number of features: 45
Assertions passed: canonical evidence covers the full Notebook 04 feature set.


## 6. Evidence Reduction — Evidence Supplied to the LLM

The full evidence has 45 transformed features. To keep the prompt compact and
focused, only the **strongest contributors** are sent to the LLM:

- top 5 positive SHAP contributors (pushing toward churn)
- top 5 negative SHAP contributors (pushing away from churn)

The selection is **programmatic** (by absolute SHAP value), never hand-specified.
The full 45-feature evidence remains available for the verifier.

> **Evidence supplied to the LLM** — this is the only evidence the LLM receives.

In [5]:
K = 5

positive = (evidence_df[evidence_df["shap_value"] > 0]
            .sort_values("shap_value", ascending=False).head(K))
negative = (evidence_df[evidence_df["shap_value"] < 0]
            .sort_values("shap_value", ascending=True).head(K))

def to_evidence_list(df):
    return [
        {
            "feature": row.feature,
            "value_display": row.value_display,
            "shap_value": float(row.shap_value),
            "direction": row.direction,
        }
        for row in df.itertuples()
    ]

reduced = {
    "positive": to_evidence_list(positive),
    "negative": to_evidence_list(negative),
}

print("Top", K, "positive SHAP contributors (toward churn):")
display(pd.DataFrame(reduced["positive"]).round(4))
print("Top", K, "negative SHAP contributors (away from churn):")
display(pd.DataFrame(reduced["negative"]).round(4))

supplied_features = {e["feature"] for e in reduced["positive"] + reduced["negative"]}
print("Features supplied to the LLM (", len(supplied_features), "):")
print(sorted(supplied_features))

Top 5 positive SHAP contributors (toward churn):


,feature,value_display,shap_value,direction
0,numerical__MonthlyCharges,106.35,0.5016,toward churn
1,categorical__Contract_Month-to-month,Month-to-month,0.3872,toward churn
2,categorical__InternetService_Fiber optic,Fiber optic,0.2191,toward churn
3,categorical__PaymentMethod_Electronic check,Electronic check,0.1986,toward churn
4,categorical__PaperlessBilling_No,not No,0.1314,toward churn


Top 5 negative SHAP contributors (away from churn):


,feature,value_display,shap_value,direction
0,numerical__tenure,34,-0.4771,away from churn
1,numerical__TotalCharges,3549.25,-0.4243,away from churn
2,categorical__OnlineBackup_No,not No,-0.0722,away from churn
3,numerical__SeniorCitizen,0,-0.0403,away from churn
4,categorical__Contract_One year,not One year,-0.0305,away from churn


Features supplied to the LLM ( 10 ):
['categorical__Contract_Month-to-month', 'categorical__Contract_One year', 'categorical__InternetService_Fiber optic', 'categorical__OnlineBackup_No', 'categorical__PaperlessBilling_No', 'categorical__PaymentMethod_Electronic check', 'numerical__MonthlyCharges', 'numerical__SeniorCitizen', 'numerical__TotalCharges', 'numerical__tenure']


## 7. LLM Provider Design

`generate_churn_explanation(payload, client, model)` centralises the API call.

- No API key is hard-coded anywhere.
- If `client is None` (no key), the function is never called.
- The model name actually used is displayed at call time.

If `OPENAI_API_KEY` is not configured, the notebook explains this and proceeds
with a clearly labelled deterministic mock so that the verification layer can
still be demonstrated end-to-end.

In [6]:
def generate_churn_explanation(payload, client, model):
    '''Call the LLM with a strict system prompt and a JSON-schema-constrained
    user payload. Returns the parsed structured response (dict).'''
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(payload, indent=2)},
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        response_format={"type": "json_object"},
    )
    content = response.choices[0].message.content
    return json.loads(content)

## 8. Strict System Instruction

The system prompt establishes the LLM as an **evidence-grounded explanation
assistant** with explicit rules.

In [7]:
SYSTEM_PROMPT = (
    "You are an evidence-grounded churn explanation assistant.\n"
    "Your job is to verbalize the supplied model evidence, not to generate new "
    "evidence.\n"
    "Rules:\n"
    "1. Use ONLY the evidence provided in the user message.\n"
    "2. Do not invent customer attributes.\n"
    "3. Do not invent numerical values.\n"
    "4. Do not invent SHAP values.\n"
    "5. Do not introduce reasons absent from the evidence.\n"
    "6. Do not infer unprovided demographic, financial, geographic, "
    "behavioral, or account information.\n"
    "7. Do not claim that any feature caused churn.\n"
    "8. Explain features as factors that pushed the MODEL prediction toward or "
    "away from churn.\n"
    "9. Preserve the direction of each SHAP contribution exactly as supplied.\n"
    "10. If evidence is insufficient, explicitly say so.\n"
    "11. Do not cite external knowledge.\n"
    "12. Do not mention information that is not present in the evidence.\n"
    "13. Report predicted_probability and threshold exactly as supplied.\n"
    "14. Respond ONLY with a JSON object of this exact shape: "
    '{"risk_summary": string, "predicted_probability": number, '
    '"threshold": number, "predicted_class": string, '
    '"supporting_evidence": [{"feature": string, "direction": string, '
    '"shap_value": number, "explanation": string}], '
    '"countervailing_evidence": [{"feature": string, "direction": string, '
    '"shap_value": number, "explanation": string}], '
    '"limitations": string}.'
)
print("System prompt defined (", len(SYSTEM_PROMPT), "chars).")

System prompt defined ( 1309 chars).


## 9. LLM Prompt Payload

The user message is built **programmatically** from the reduced evidence. It
contains:

- predicted probability
- decision threshold
- predicted class
- positive evidence (top 5)
- negative evidence (top 5)

It does **not** contain:

- the true target label (it is not even present in the evidence JSON),
- Notebook 02 statistical results,
- model performance metrics (holdout F1/ROC/etc.),
- any unrelated raw customer information.

The LLM explains the prediction; it is never told whether the prediction was
correct.

In [8]:
def build_llm_payload(reduced, authoritative):
    predicted_class = "Churn" if authoritative["predicted_class"] == 1 else "Retained"
    return {
        "task": ("Explain the model's churn prediction for this customer "
                 "using ONLY the evidence below."),
        "predicted_probability": authoritative["predicted_probability"],
        "threshold": authoritative["threshold"],
        "predicted_class": predicted_class,
        "supporting_evidence": reduced["positive"],
        "countervailing_evidence": reduced["negative"],
    }


llm_payload = build_llm_payload(reduced, authoritative)

print("LLM prompt payload keys:", list(llm_payload.keys()))
print("Ground truth / performance metrics present in payload:",
      any(k in json.dumps(llm_payload).lower() for k in
          ["actual", "true label", "holdout", "f1", "accuracy", "roc"]))
print("Predicted probability in payload:", llm_payload["predicted_probability"])
print("Threshold in payload:", llm_payload["threshold"])
print("Supporting evidence features in payload:",
      [e["feature"] for e in llm_payload["supporting_evidence"]])

LLM prompt payload keys: ['task', 'predicted_probability', 'threshold', 'predicted_class', 'supporting_evidence', 'countervailing_evidence']
Ground truth / performance metrics present in payload: False
Predicted probability in payload: 0.7300524711608887
Threshold in payload: 0.525
Supporting evidence features in payload: ['numerical__MonthlyCharges', 'categorical__Contract_Month-to-month', 'categorical__InternetService_Fiber optic', 'categorical__PaymentMethod_Electronic check', 'categorical__PaperlessBilling_No']


## 10. Deterministic Verification Layer

The verifier is **independent of the LLM**. It inspects the generated
structured output and compares every claim against the authoritative evidence.

It flags:

- **A. Unknown feature** — a feature not present in the authoritative evidence.
- **B. Fabricated SHAP value** — a SHAP value that differs from the
  authoritative value (tolerance `atol=1e-6`).
- **C. Wrong direction** — a positive SHAP described as "away from churn" or a
  negative SHAP as "toward churn".
- **D. Fabricated probability** — a probability different from the
  authoritative prediction.
- **E. Fabricated threshold** — a threshold different from the authoritative
  threshold.
- **F. Unsupported evidence** — a reason/feature absent from the **supplied**
  (reduced) evidence.
- **G. Causal language** — statements claiming a feature "caused" churn.
- **H. Ground-truth leakage** — reference to the actual target outcome, which
  is never supplied to the LLM.

A single failed check flips the status to `FLAG`.

In [9]:
CAUSAL_PATTERNS = [
    "caused churn", "causes churn", "cause churn",
    "caused the customer to churn", "will cause churn",
    "led to churn", "because of",
]

LEAKAGE_PATTERNS = [
    "actually churned", "did in fact churn", "did churn",
    "true label", "ground truth", "was correct", "correctly predicted",
]

ALLOWED_DIRECTIONS = {"toward churn", "away from churn"}


def direction_of(shap_value):
    return "toward churn" if shap_value >= 0 else "away from churn"


def verify_explanation(generated, authoritative, supplied_features):
    '''Deterministically check a generated structured explanation against the
    authoritative evidence. Returns {status, checks, flags}.'''
    checks = {}
    flags = []

    def record(check_name, ok, reason):
        checks[check_name] = bool(ok)
        if not ok:
            flags.append(reason)

    auth_by_feature = {f["feature"]: f for f in authoritative["features"]}
    expected_class = "Churn" if authoritative["predicted_class"] == 1 else "Retained"

    # D. Probability grounding
    gen_prob = generated.get("predicted_probability")
    ok = (isinstance(gen_prob, (int, float)) and np.isfinite(gen_prob)
          and np.isclose(float(gen_prob), authoritative["predicted_probability"], atol=1e-6))
    record("probability_grounded", ok,
           f"Fabricated probability: got {gen_prob!r}, authoritative "
           f"{authoritative['predicted_probability']}")

    # E. Threshold grounding
    gen_thr = generated.get("threshold")
    ok = (isinstance(gen_thr, (int, float)) and np.isfinite(gen_thr)
          and np.isclose(float(gen_thr), authoritative["threshold"], atol=1e-6))
    record("threshold_grounded", ok,
           f"Fabricated threshold: got {gen_thr!r}, authoritative "
           f"{authoritative['threshold']}")

    # Predicted-class grounding (extra, beyond the minimum set)
    ok = generated.get("predicted_class") == expected_class
    record("predicted_class_grounded", ok,
           f"Predicted class mismatch: got {generated.get('predicted_class')!r}, "
           f"expected {expected_class!r}")

    items = list(generated.get("supporting_evidence", [])) + \
            list(generated.get("countervailing_evidence", []))

    # A/F. Feature grounding + unsupported evidence
    features_grounded = True
    no_unsupported = True
    for item in items:
        fname = item.get("feature")
        if fname not in auth_by_feature:
            features_grounded = False
            flags.append(f"Unknown feature referenced: {fname!r}")
        if fname not in supplied_features:
            no_unsupported = False
            flags.append(f"Unsupported evidence: {fname!r} not in the supplied evidence set")
    checks["features_grounded"] = bool(features_grounded)
    checks["no_unsupported_evidence"] = bool(no_unsupported)

    # B. SHAP value grounding
    shap_grounded = True
    for item in items:
        fname = item.get("feature")
        if fname not in auth_by_feature:
            continue
        auth = auth_by_feature[fname]
        gen_sv = item.get("shap_value")
        ok = (isinstance(gen_sv, (int, float)) and np.isfinite(gen_sv)
              and np.isclose(float(gen_sv), auth["shap_value"], atol=1e-6))
        if not ok:
            shap_grounded = False
            flags.append(f"Fabricated SHAP for {fname!r}: got {gen_sv!r}, "
                         f"authoritative {auth['shap_value']:.6f}")
    checks["shap_values_grounded"] = bool(shap_grounded)

    # C. Direction grounding
    directions_ok = True
    for item in items:
        fname = item.get("feature")
        if fname not in auth_by_feature:
            continue
        expected = direction_of(auth_by_feature[fname]["shap_value"])
        if item.get("direction") != expected:
            directions_ok = False
            flags.append(f"Wrong direction for {fname!r}: got "
                         f"{item.get('direction')!r}, expected {expected!r}")
    checks["directions_consistent"] = bool(directions_ok)

    # G. Causal language
    texts = [str(generated.get("risk_summary", "")), str(generated.get("limitations", ""))]
    texts += [str(item.get("explanation", "")) for item in items]
    blob = " ".join(texts).lower()
    causal_hits = [p for p in CAUSAL_PATTERNS if p in blob]
    checks["no_causal_claim"] = not causal_hits
    if causal_hits:
        flags.append(f"Potential causal claim detected: {causal_hits}")

    # H. Ground-truth leakage
    leak_hits = [p for p in LEAKAGE_PATTERNS if p in blob]
    checks["no_ground_truth_leakage"] = not leak_hits
    if leak_hits:
        flags.append(f"Potential ground-truth leakage detected: {leak_hits}")

    status = "PASS" if not flags else "FLAG"
    return {"status": status, "checks": checks, "flags": flags}

## 11. Adversarial Test (Required)

To prove the verifier actually detects fabrication, an **intentionally
incorrect** explanation is constructed deterministically in Python. It contains
at least:

- one nonexistent feature (`customer_income`),
- one incorrect SHAP value,
- one reversed direction,
- one incorrect probability,
- one incorrect threshold,
- causal language,
- ground-truth leakage.

The verifier must return **FLAG** and name every failing check. No LLM is used
to construct this example.

In [10]:
def build_adversarial_explanation(authoritative):
    '''Deterministically fabricated explanation to stress-test the verifier.'''
    mcm = next(f for f in authoritative["features"]
               if f["feature"] == "numerical__MonthlyCharges")
    return {
        "risk_summary": "High monthly charges led to churn for this customer.",
        "predicted_probability": 0.9999,
        "threshold": 0.50,
        "predicted_class": "Churn",
        "supporting_evidence": [
            {
                "feature": "customer_income",
                "direction": "toward churn",
                "shap_value": 0.9,
                "explanation": "High customer income caused churn.",
            },
            {
                "feature": mcm["feature"],
                "direction": "away from churn",   # reversed: authoritative is +0.5016
                "shap_value": 0.1234,             # wrong value
                "explanation": "The customer's monthly charge pushed the model "
                               "away from churn.",
            },
        ],
        "countervailing_evidence": [],
        "limitations": "The customer actually churned, so this is the ground truth.",
    }


adversarial = build_adversarial_explanation(authoritative)
adv_result = verify_explanation(adversarial, authoritative, supplied_features)

print("Adversarial verification status:", adv_result["status"])
print("Checks:", json.dumps(adv_result["checks"], indent=2))
print("Flags:")
for f in adv_result["flags"]:
    print("  -", f)

assert adv_result["status"] == "FLAG", "Adversarial example must be flagged"
print("\nAssertion passed: adversarial explanation correctly produced FLAG.")

Adversarial verification status: FLAG
Checks: {
  "probability_grounded": false,
  "threshold_grounded": false,
  "predicted_class_grounded": true,
  "features_grounded": false,
  "no_unsupported_evidence": false,
  "shap_values_grounded": false,
  "directions_consistent": false,
  "no_causal_claim": false,
  "no_ground_truth_leakage": false
}
Flags:
  - Fabricated probability: got 0.9999, authoritative 0.7300524711608887
  - Fabricated threshold: got 0.5, authoritative 0.525
  - Unknown feature referenced: 'customer_income'
  - Unsupported evidence: 'customer_income' not in the supplied evidence set
  - Fabricated SHAP for 'numerical__MonthlyCharges': got 0.1234, authoritative 0.501590
  - Wrong direction for 'numerical__MonthlyCharges': got 'away from churn', expected 'toward churn'
  - Potential causal claim detected: ['caused churn', 'led to churn']
  - Potential ground-truth leakage detected: ['actually churned', 'ground truth']

Assertion passed: adversarial explanation correctly

## 12. Valid Evidence-Compliant Example (Deterministic Mock)

A fully evidence-compliant response is constructed deterministically from the
authoritative evidence. It is used to show the verifier returning **PASS**.

> **Mock response — not an LLM result.**

This mock is constructed only from the supplied evidence and is used to prove
the PASS path; it is never presented as actual AI output.

In [11]:
def build_mock_explanation(authoritative, reduced):
    expected_class = "Churn" if authoritative["predicted_class"] == 1 else "Retained"
    supporting = [
        {
            "feature": e["feature"],
            "direction": e["direction"],
            "shap_value": e["shap_value"],
            "explanation": (
                f"The model's output was pushed toward churn by {e['feature']} "
                f"(value {e['value_display']})."
            ),
        }
        for e in reduced["positive"]
    ]
    countervailing = [
        {
            "feature": e["feature"],
            "direction": e["direction"],
            "shap_value": e["shap_value"],
            "explanation": (
                f"The model's output was pushed away from churn by {e['feature']} "
                f"(value {e['value_display']})."
            ),
        }
        for e in reduced["negative"]
    ]
    return {
        "risk_summary": (
            f"The model assigns this customer a "
            f"{authoritative['predicted_probability'] * 100:.2f}% churn probability, "
            f"above the {authoritative['threshold'] * 100:.1f}% decision threshold."
        ),
        "predicted_probability": authoritative["predicted_probability"],
        "threshold": authoritative["threshold"],
        "predicted_class": expected_class,
        "supporting_evidence": supporting,
        "countervailing_evidence": countervailing,
        "limitations": (
            "This explanation is derived solely from the supplied model "
            "evidence and does not establish causality."
        ),
    }


mock_explanation = build_mock_explanation(authoritative, reduced)
mock_result = verify_explanation(mock_explanation, authoritative, supplied_features)

print("Mock response verification status:", mock_result["status"])
print("Checks:", json.dumps(mock_result["checks"], indent=2))
print("Flags:", mock_result["flags"])

assert mock_result["status"] == "PASS", "Evidence-compliant mock must pass"
print("\nAssertion passed: evidence-compliant mock produced PASS.")

Mock response verification status: PASS
Checks: {
  "probability_grounded": true,
  "threshold_grounded": true,
  "predicted_class_grounded": true,
  "features_grounded": true,
  "no_unsupported_evidence": true,
  "shap_values_grounded": true,
  "directions_consistent": true,
  "no_causal_claim": true,
  "no_ground_truth_leakage": true
}
Flags: []

Assertion passed: evidence-compliant mock produced PASS.


## 13. Evidence → Explanation → Verification Demonstration

This is the end-to-end path:

```
AUTHORITATIVE EVIDENCE
        ↓
LLM EXPLANATION
        ↓
DETERMINISTIC VERIFIER
        ↓
PASS / FLAG
```

If `OPENAI_API_KEY` is configured, a real LLM call is made with the selected
model and its output is verified. Otherwise (as in this run, if no key is set),
the clearly labelled deterministic mock from section 12 is used as the
"generated explanation" and verified the same way. **No mock is ever presented
as a real LLM result.**

In [12]:
if client is not None:
    generated = generate_churn_explanation(llm_payload, client, OPENAI_MODEL)
    generated_source = f"LLM-generated ({OPENAI_MODEL})"
    print("Real LLM call executed with model:", OPENAI_MODEL)
else:
    generated = build_mock_explanation(authoritative, reduced)
    generated_source = "Mock response (no OPENAI_API_KEY configured) - not an LLM result"
    print("No OPENAI_API_KEY configured. Using deterministic evidence-compliant mock.")
    print("The mock is constructed from the authoritative evidence and is used "
          "only to exercise the verification layer.")

print("\nGenerated explanation source:", generated_source)

No OPENAI_API_KEY configured. Using deterministic evidence-compliant mock.
The mock is constructed from the authoritative evidence and is used only to exercise the verification layer.

Generated explanation source: Mock response (no OPENAI_API_KEY configured) - not an LLM result


### Structured response (raw)

The structured JSON returned by the explanation step (mock in this run, LLM
when a key is present).

In [13]:
print(json.dumps(generated, indent=2))

{
  "risk_summary": "The model assigns this customer a 73.01% churn probability, above the 52.5% decision threshold.",
  "predicted_probability": 0.7300524711608887,
  "threshold": 0.525,
  "predicted_class": "Churn",
  "supporting_evidence": [
    {
      "feature": "numerical__MonthlyCharges",
      "direction": "toward churn",
      "shap_value": 0.5015904307365417,
      "explanation": "The model's output was pushed toward churn by numerical__MonthlyCharges (value 106.35)."
    },
    {
      "feature": "categorical__Contract_Month-to-month",
      "direction": "toward churn",
      "shap_value": 0.38723093271255493,
      "explanation": "The model's output was pushed toward churn by categorical__Contract_Month-to-month (value Month-to-month)."
    },
    {
      "feature": "categorical__InternetService_Fiber optic",
      "direction": "toward churn",
      "shap_value": 0.21912768483161926,
      "explanation": "The model's output was pushed toward churn by categorical__InternetSe

### Verification of the generated explanation

The same deterministic verifier is applied to the actual generated response.

In [14]:
generated_result = verify_explanation(generated, authoritative, supplied_features)

print("Generated explanation verification status:", generated_result["status"])
print("Checks:", json.dumps(generated_result["checks"], indent=2))
print("Flags:", generated_result["flags"])

assert generated_result["status"] == "PASS"
print("\nAssertion passed: generated (mock) explanation is evidence-compliant.")

Generated explanation verification status: PASS
Checks: {
  "probability_grounded": true,
  "threshold_grounded": true,
  "predicted_class_grounded": true,
  "features_grounded": true,
  "no_unsupported_evidence": true,
  "shap_values_grounded": true,
  "directions_consistent": true,
  "no_causal_claim": true,
  "no_ground_truth_leakage": true
}
Flags: []

Assertion passed: generated (mock) explanation is evidence-compliant.


## 14. Pipeline Summary

```
        AUTHORITATIVE EVIDENCE (SHAP JSON from Notebook 04)
                          |
                          v
        Evidence reduction (top 5 positive + top 5 negative)
                          |
                          v
        LLM explanation (constrained by strict system prompt)
                          |
                          v
        Deterministic verifier (independent of the LLM)
                          |
                          v
                        PASS / FLAG
```

The LLM is **not trusted as an evidence source**. It receives a constrained
evidence set derived from the model's SHAP output, and every structured claim
it returns is checked against the independently stored authoritative evidence.
In this run:

- Adversarial (fabricated) explanation → **FLAG** (all fabricated claims
  detected).
- Evidence-compliant explanation → **PASS**.
- The verifier ran deterministically, with no LLM involvement in any check.

If a real LLM call were made, its output would be verified identically; the
verifier does not care where the explanation came from.

## 15. Limitations

The verifier is intentionally lightweight and this must be stated honestly:

- It validates **structured claims and known feature references**. It does not
  provide formal semantic proof that every sentence is truthful.
- An LLM could phrase an unsupported claim **without mentioning an explicit
  feature name** (e.g., a fabricated narrative fact); the verifier would not
  necessarily catch it.
- Keyword-based causal-language detection is imperfect: it may miss a causal
  claim phrased with different words, or (rarely) over-flag ordinary phrasing.
- Structured JSON output improves validation but does not eliminate all
  possible hallucinations.
- The system therefore **reduces and detects** several classes of unsupported
  claims; it does not mathematically guarantee hallucination-free natural
  language.

Security notes:

- The API key is read from `OPENAI_API_KEY` and is never printed, committed, or
  written to any file (including the evidence JSON).
- `OPENAI_MODEL` controls the model with a documented default; the exact model
  used is printed at call time.
- `.env` files are ignored via `.gitignore`, so a local `.env` never enters Git
  history.